# Rail_Corrugation: train and evaluate balanced logistic regression (C=1)

Run the cells in order, or select **Run All**. This notebook loads the raw Train dataset, builds features, trains new models, and prints the measured scores. All implementation is in this notebook. No saved model or result file is required.

**Local data:** copy `Rail_Corrugation` from the supplied `PS3/02_Datasets` bundle into this folder's `data/`, retaining the Train names below. Data stays local and is ignored by Git.

```text
Rail_Corrugation/
  train.ipynb
  data/
    Train_Labels.csv
    Train/
      Train1.csv ... Train272.csv
```

Use Python 3.11. Install the pinned CPU libraries once in the notebook's Python environment:
```python
%pip install numpy==1.26.4 pandas==2.2.1 scipy==1.12.0 scikit-learn==1.4.1.post1 openpyxl==3.1.5 rainflow==3.2.0 threadpoolctl==3.4.0
```

The displayed scores are cross-validation on labelled **Train** data. The recipe was selected in earlier experiments on this corpus, so these are exploratory validation results. Official Test is never read. Exact duplicate files stay in the same fold. Unknown source-run relationships and only 14 Side I examples limit generalization. Raw feature extraction reads about 4.4 GB and may take several minutes.

## 1. Imports and data location

In [1]:
from pathlib import Path
import sys, time, hashlib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits
HERE = Path.cwd() if Path.cwd().name == 'Rail_Corrugation' else Path.cwd()/'Rail_Corrugation'
DATA = HERE/'data'
assert DATA.is_dir(), f'Place the Rail_Corrugation Train dataset in {DATA} first (see the cell above).'
SEED = 17
started = time.perf_counter()
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)
print('Data:', DATA.resolve())
from scipy.signal import welch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

Python: 3.11.4 | NumPy: 1.26.4 | pandas: 2.2.1 | sklearn: 1.4.1.post1
Data: C:\000NebulaX\nebulax_p3\Rail_Corrugation\data


## 2. Extract side-aware features from one raw file at a time
The retained extractor computes amplitude and Welch spectrum summaries, keeping Side I and Side II separate. Tachometer values are checked but excluded from the model.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import welch
BANDS = ((0, 100), (100, 300), (300, 800), (800, 1600), (1600, 3000), (3000, 5000))

def extract(path: Path) -> tuple[dict, dict]:
    frame = pd.read_csv(path, header=0, dtype=np.float64)
    if frame.shape != (10000, 129) or frame.columns[0] != "Rotating speed":
        raise ValueError(f"Unexpected shape/header for {path.name}: {frame.shape}")
    expected = ["Rotating speed"] + [
        f"{kind} of bearing in position {position} of car {car}"
        for car in range(1, 9) for position in range(1, 9) for kind in ("Vibration", "Shock")
    ]
    if list(frame.columns) != expected:
        raise ValueError(f"Channel order mismatch: {path.name}")
    values = frame.to_numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite data: {path.name}")
    tach = values[:, 0]
    qc = {"file_id": path.name, "n_rows": len(values), "tach_constant": bool(np.ptp(tach) == 0),
          "tach_constant_one": bool(np.all(tach == 1)),
          "tach_quality": "unknown_invalid" if np.ptp(tach) == 0 else "varying_unconverted",
          "tach_in_model": False}
    x = values[:, 1:]
    rms = np.sqrt(np.mean(x * x, axis=0))
    std = x.std(axis=0)
    absx = np.abs(x)
    quantiles = np.quantile(absx, (0.5, 0.95, 0.99), axis=0)
    centered = x - x.mean(axis=0)
    kurt = np.mean(centered**4, axis=0) / np.maximum(std**4, 1e-24)
    crest = np.max(absx, axis=0) / np.maximum(rms, 1e-12)
    frequencies, density = welch(x, fs=10000.0, window="hann", nperseg=1024,
                                 noverlap=512, detrend="constant", scaling="density", axis=0)
    power = density * (frequencies[1] - frequencies[0])
    total = power.sum(axis=0)
    shape = power / np.maximum(total, 1e-24)
    centroid = (shape * frequencies[:, None]).sum(axis=0)
    entropy = -(shape * np.log(np.maximum(shape, 1e-24))).sum(axis=0) / np.log(len(frequencies))
    channel_stats = {
        "rms": rms, "std": std, "abs_q50": quantiles[0], "abs_q95": quantiles[1],
        "abs_q99": quantiles[2], "crest": crest, "kurtosis": kurt,
        "spectral_centroid_hz": centroid, "spectral_entropy": entropy,
    }
    for lo, hi in BANDS:
        mask = (frequencies >= lo) & ((frequencies < hi) if hi < 5000 else (frequencies <= hi))
        band_power = power[mask].sum(axis=0)
        channel_stats[f"log1p_band_power_{lo}_{hi}hz"] = np.log1p(band_power)
        channel_stats[f"band_fraction_{lo}_{hi}hz"] = band_power / np.maximum(total, 1e-24)
    # Exact map: car, position, modality; even zero-based position means Side I.
    stats = {name: value.reshape(8, 8, 2) for name, value in channel_stats.items()}
    result = {}
    for stat, array in stats.items():
        for modality, modality_name in enumerate(("vibration", "shock")):
            side_values = []
            for side in (0, 1):
                selected = array[:, side::2, modality]
                mean = float(np.mean(selected))
                side_values.append(mean)
                prefix = f"{modality_name}_side{side+1}_{stat}"
                result[f"{prefix}_mean"] = mean
                result[f"{prefix}_max"] = float(np.max(selected))
                for car in range(8):
                    result[f"car{car+1}_{prefix}_mean"] = float(np.mean(selected[car]))
            first, second = side_values
            result[f"{modality_name}_{stat}_side_difference"] = first - second
            result[f"{modality_name}_{stat}_normalized_side_difference"] = (first - second) / (abs(first) + abs(second) + 1e-12)
    if not np.isfinite(list(result.values())).all():
        raise ValueError(f"Nonfinite features for {path.name}")
    return result, qc

## 3. Load labels, extract features, and group exact duplicates

In [3]:
labels=pd.read_csv(DATA/'Train_Labels.csv')
names=labels.filename.tolist()
y=labels.label.to_numpy()
rows,hashes=[],[]
for i,name in enumerate(names,1):
    assert Path(name).name==name
    path=DATA/'Train'/name
    hashes.append(hashlib.sha256(path.read_bytes()).hexdigest())
    values,quality=extract(path)
    rows.append(values)
    if i%20==0 or i==len(names): print(f'Extracted {i}/{len(names)} files',flush=True)
feature_table=pd.DataFrame(rows)
X=feature_table.to_numpy(float);groups=np.asarray(hashes)
assert np.isfinite(X).all()
for group in set(groups): assert len(set(y[groups==group]))==1
print('Files:',len(names),'| Unique file groups:',len(set(groups)),'| Features:',X.shape[1])
display(labels.label.value_counts().rename('files').to_frame())

Extracted 20/272 files


Extracted 40/272 files


Extracted 60/272 files


Extracted 80/272 files


Extracted 100/272 files


...

Extracted 240/272 files


Extracted 260/272 files


Extracted 272/272 files


Files: 272 | Unique file groups: 270 | Features: 924


,files
label,
Normal,234
Side II,24
Side I,14


## 4. Train five stratified group folds
Whole-file SHA-256 groups keep duplicates together. Standardization and class-balanced logistic regression are fitted independently inside each fitting fold.

In [4]:
LABELS=['Normal','Side I','Side II']
def estimator():
    return make_pipeline(StandardScaler(),LogisticRegression(C=1,class_weight='balanced',
                          solver='lbfgs',max_iter=3000,random_state=SEED))
def macro_f1(truth,prediction):
    return f1_score(truth,prediction,labels=LABELS,average='macro',zero_division=0)
oof=np.empty(len(y),dtype=object);fold_rows=[]
splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
for fold,(fit,val) in enumerate(splitter.split(X,y,groups),1):
    assert not set(groups[fit])&set(groups[val])
    with threadpool_limits(limits=2):
        model=estimator().fit(X[fit],y[fit])
        fit_pred=model.predict(X[fit]);oof[val]=model.predict(X[val])
    row={'fold':fold,'train_fit':macro_f1(y[fit],fit_pred),'validation':macro_f1(y[val],oof[val]),'held_out_files':len(val)}
    fold_rows.append(row)
    print(f"Fold {fold}: train={row['train_fit']:.6f}, validation={row['validation']:.6f}")
display(pd.DataFrame(fold_rows))

Fold 1: train=1.000000, validation=0.776852
Fold 2: train=1.000000, validation=0.808038
Fold 3: train=1.000000, validation=0.826667


Fold 4: train=1.000000, validation=0.802047
Fold 5: train=1.000000, validation=0.597771


,fold,train_fit,validation,held_out_files
0,1,1.0,0.776852,55
1,2,1.0,0.808038,55
2,3,1.0,0.826667,55
3,4,1.0,0.802047,53
4,5,1.0,0.597771,54


## 5. Print validation results and fit the final model

In [5]:
validation_score=macro_f1(y,oof)
train_score=np.mean([r['train_fit'] for r in fold_rows])
print(f'Mean fitting-fold macro F1: {train_score:.9f}')
print(f'Out-of-fold macro F1: {validation_score:.9f}')
print(f'Always-Normal baseline: {macro_f1(y,np.repeat("Normal",len(y))):.9f}')
print(classification_report(y,oof,labels=LABELS,zero_division=0,digits=4))
display(pd.DataFrame(confusion_matrix(y,oof,labels=LABELS),index=LABELS,columns=LABELS))
with threadpool_limits(limits=2): final_model=estimator().fit(X,y)
print('Final model trained on all Train files; available as final_model.')
print(f'Elapsed: {time.perf_counter()-started:.1f} seconds')

Mean fitting-fold macro F1: 1.000000000
Out-of-fold macro F1: 0.757523278
Always-Normal baseline: 0.308300395
              precision    recall  f1-score   support

      Normal     0.9540    0.9744    0.9641       234
      Side I     0.6000    0.4286    0.5000        14
     Side II     0.8261    0.7917    0.8085        24

    accuracy                         0.9301       272
   macro avg     0.7934    0.7315    0.7575       272
weighted avg     0.9245    0.9301    0.9264       272



,Normal,Side I,Side II
Normal,228,3,3
Side I,7,6,1
Side II,4,1,19


Final model trained on all Train files; available as final_model.
Elapsed: 128.1 seconds
